# Regression Modeling in Pyhon: Linear Regression

This notebook provides a hands-on introduction to **linear regression**, a foundational statistical modeling technique. We will explora how to build, interpret, and visualize linear regression models using Python. Instead of using the NHANES dataset which is used in the related notebook in the course, we will work with a classic dataset containing information about tips given in a restaurant. This allows us to explore the same statistical concepts with a fresh and interesting dataset, without risking to face any copyright issues.

---

## What is Regression? 
Regression analysis is a set of statistical methods used to estimate the relationships between a dependent variable (often called the **outcome**) and one or more independent variables (often called **predictors** or **covariates**). Linear regression, specifically, models this relationship by fitting a linear equation to the observed data.

We will be using popular Python libraries for our analysis:

* **Pandas:** For data manipulation and analysis.
* **Numpy:** For numerical operations.
* **Statsmodels:** For fitting and evaluating statistical models.
* **Matplotlib & Seaborn:** For data visualization.

Let's start by importing these libraries.

In [3]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

### Loading the Data

We will use the 'tips' dataset, which is conveniently available through the Seaborn library. This dataset contains records of tips given by diners, along with information about the totall bill, the diner's gender, whether they were a smoker, the day of the week, the time of the day, and the size of their party. We will start by loading the data and examining its structure. 

In [4]:
# Load the 'tips' dataset from Seaborn
tips = sns.load_dataset("tips")

# Display the first few rows to understand its structure
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


---

## Simple Linear Regression: Predicting Tips from the Totall Bill

We'll begin our exploration with a simple linear regression model. Our goal is to predict the `tip` amount based on a single predictor: the `total_bill`. The `tip` is our outcome (or dependent) variable, and `total_bill` is our predictor (or independent) variable. 

The model we are fitting can be expressed as:

`E[tip] = Intercept + Slope * total_bill`

This formula states that the expected tip is a linear function of the total bill. Let's use `statsmodels` to fit this model.

In [10]:
model = sm.OLS.from_formula("tip ~ total_bill", data=tips)
result = model.fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                  0.454
Method:                 Least Squares   F-statistic:                     203.4
Date:                Thu, 16 Oct 2025   Prob (F-statistic):           6.69e-34
Time:                        12:14:03   Log-Likelihood:                -350.54
No. Observations:                 244   AIC:                             705.1
Df Residuals:                     242   BIC:                             712.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9203      0.160      5.761      0.0

### Interpreting the Model Parameters

Let's focus on the coefficients table in the summary above:

|               | coef   | std err | t      | P>\|t\| | [0.025 | 0.975] |
|---------------|--------|---------|--------|-------|--------|--------|
| **Intercept** | 0.9203 | 0.160   | 5.761  | 0.000 | 0.606  | 1.235  |
| **total_bill**| 0.1050 | 0.007   | 14.260 | 0.000 | 0.091  | 0.119  |

* **Intercept (0.9203):** This is the predicted tip amount when the `total_bill` is zero. In this context, it doesn't have a practival meaning (a 0$ bill won't get a tip), but it's a necessary part of the model that anchors the regression line.
* **total_bill_coefficient (0.1050):** This is the most interesting part. It's the _slope_ of our line. It means that for every additional dollar on the `total_bill`, we expect the `tip` to increase by approximately 10.5 cents.

The p-value (`P>|t|`) for `total_bill` is 0.000, which is very small. This indicates that there is a statistically significant relationship between the total bill and the tip amount.

### Understanging the R-squared

The **R-squared** value in the summary is 0.457. This statistic tells us the proportion of the variance in the outcome variable (`tip`) that is predictable from the independent variable (`total_bill`). In our case, 45.7% of the variability in tips can be explained by the total bill amount. This is a moderately strong relationship.

For a simple linear regression with one predictor, the R-squared is simply the square of the Pearson correlation coefficient between the predictor and the outcome.

In [7]:
corr = tips[["tip", "total_bill"]].corr()
print(f"Squared correlation: {corr.tip.total_bill**2:.3f}")

Squared correlation: 0.457


---

## Multiple Linear Regression: Adding More Predictors

The real power of regression comes from including multiple predictors. Let's add the `size` of the party (number of people) to our model. Does the size of the group affect the tip, even after accounting for the bill?

The new model is: `E[tip] = Intercept + Slope_1 * total_bill + Slope_2 * size`

In [9]:
model = sm.OLS.from_formula("tip ~ total_bill + size", data=tips)
result = model.fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.468
Model:                            OLS   Adj. R-squared:                  0.463
Method:                 Least Squares   F-statistic:                     105.9
Date:                Thu, 16 Oct 2025   Prob (F-statistic):           9.67e-34
Time:                        12:13:49   Log-Likelihood:                -347.99
No. Observations:                 244   AIC:                             702.0
Df Residuals:                     241   BIC:                             712.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.6689      0.194      3.455      0.0

### Interpreting the New Model

Now we have two coefficients to interpret:

* **total_bill (0.0927):** For every one-dollar increase in the bill, the tip is expected to increase by 9.3 cents, **holding the part size constant**.
* **size (0.1926):** For every additional person in the party, the tip is expected to increase by 19.3 cents, **holding the total bill constant**.

This concept of "holding other variables constant" is crucial in multiple regression. Each coefficient represents the unique contribution of its variable.

Notice that the coefficient for `total_bill` changed slightly (from 0.1050 to 0.0927) after we added `size` to the model. This happens because `total_bill` and `size` are correlated. When predictors are correlated, their coefficients can change when others are added or removed from the model.

The **R-squared** increased to 0.468. This means our new model explains a slightly larger proportion of the variability in tips. Adding `size` has improved our model's explanatory power, although not by a large amount.

## Adding a Categorical Variable

What about categorical variables, like whether the diner is a smoker? Let's add the `smoker` variable to our model.

When we include a categorical variable, `statsmodels` automaticall converts it into 'dummy variables'. If a variable has two levels (e.g., 'Yes' and 'No' for `smoker`), one level is chosen as the 'reference level' (its effect is absorbed into the intercept), and a coefficient is estimated for the other level.

In [13]:
model = sm.OLS.from_formula("tip ~ total_bill + size + smoker", data=tips)
result = model.fit()
print(result.summary())
print("-"*80)

                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.469
Model:                            OLS   Adj. R-squared:                  0.462
Method:                 Least Squares   F-statistic:                     70.57
Date:                Thu, 16 Oct 2025   Prob (F-statistic):           9.41e-33
Time:                        12:27:37   Log-Likelihood:                -347.80
No. Observations:                 244   AIC:                             703.6
Df Residuals:                     240   BIC:                             717.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        0.6256      0.207      3.026   

### Interpreting the Categorical Coefficient

In the output, you'll see a new coefficient: `smoker[T.No]`. This means `statsmodels` chose the 'Yes' category (smokers) as the reference level.

* **smoker[T.No] (-0.0633):** This means that a non-smoking party is expected to tip about 6.3 cents _less_ than a smoking party, holding the total bill and party size constant.

However, look at the p-value (`P>|t|`) for this coefficient: it's 0.613. Since this is much larger than the conventional significance level of 0.05, we conclude that there is **no statistically significant difference** in tipping between smokers and non-smokers after accounting for the bill size and party size. The R-squared value also barely changed, reinforcing that `smoker` is not a strong predictor in this model. 